# 5. Sonuçların Analizi

Bu notebook'ta model sonuçlarının detaylı analizini yapacağız.

**İçerik:**
- Confusion Matrix
- ROC Eğrileri
- Feature Importance
- Sonuç tabloları ve grafikler

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report, 
                             roc_curve, auc, precision_recall_curve)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print('Kütüphaneler yüklendi!')

## 5.1 Verilerin ve Modellerin Yüklenmesi

In [ ]:
DATA_PATH = '../data/'

# Veri setini yükle
df = pd.read_pickle(DATA_PATH + 'model_veri_seti.pkl')

# Özellik listesi
with open(DATA_PATH + 'ozellik_listesi.json', 'r', encoding='utf-8') as f:
    ozellik_bilgi = json.load(f)

hedef = ozellik_bilgi['hedef']
ozellikler = ozellik_bilgi['ozellikler']

# Sonuçlar
sonuc_df = pd.read_pickle(DATA_PATH + 'model_sonuclari.pkl')

print("Veriler yüklendi!")
display(sonuc_df)

In [ ]:
# Modelleri yükle
import os

modeller = {}
model_dosyalari = [f for f in os.listdir(DATA_PATH) if f.endswith('_model.pkl')]

for dosya in model_dosyalari:
    model_adi = dosya.replace('_model.pkl', '').replace('_', ' ').title()
    modeller[model_adi] = joblib.load(DATA_PATH + dosya)
    print(f"Yüklendi: {model_adi}")

In [ ]:
# Test verisini hazırla
X = df[ozellikler]
y = df[hedef]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Ölçeklendirme
scaler = joblib.load(DATA_PATH + 'scaler.pkl')
X_test_scaled = scaler.transform(X_test)

print(f"Test seti boyutu: {X_test.shape}")

## 5.2 Confusion Matrix Görselleştirmesi

In [ ]:
def ciz_confusion_matrix(y_true, y_pred, model_adi, ax):
    """Confusion matrix çizer."""
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Yıldırım Yok', 'Yıldırım Var'],
                yticklabels=['Yıldırım Yok', 'Yıldırım Var'])
    ax.set_xlabel('Tahmin')
    ax.set_ylabel('Gerçek')
    ax.set_title(f'{model_adi}')

# Tüm modeller için confusion matrix
n_models = len(modeller)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (model_adi, model) in enumerate(modeller.items()):
    if idx >= 4:
        break
    
    # Tahmin yap
    if 'Lojistik' in model_adi:
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)
    
    ciz_confusion_matrix(y_test, y_pred, model_adi, axes[idx])

plt.tight_layout()
plt.savefig('../tez_docs/figures/confusion_matrices.png', dpi=150)
plt.show()

## 5.3 ROC Eğrileri

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

renkler = ['blue', 'green', 'red', 'orange']

for idx, (model_adi, model) in enumerate(modeller.items()):
    # Tahmin olasılıkları
    if 'Lojistik' in model_adi:
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = model.predict_proba(X_test)[:, 1]
    
    # ROC eğrisi
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    
    ax.plot(fpr, tpr, color=renkler[idx % len(renkler)], lw=2,
            label=f'{model_adi} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Rastgele (AUC = 0.500)')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Yanlış Pozitif Oranı (1 - Özgüllük)')
ax.set_ylabel('Doğru Pozitif Oranı (Duyarlılık)')
ax.set_title('ROC Eğrileri Karşılaştırması')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../tez_docs/figures/roc_curves.png', dpi=150)
plt.show()

## 5.4 Feature Importance (Random Forest)

In [ ]:
# Random Forest feature importance
if 'Random Forest' in modeller:
    rf_model = modeller['Random Forest']
    
    # Feature importance
    importance = pd.DataFrame({
        'ozellik': ozellikler,
        'onem': rf_model.feature_importances_
    }).sort_values('onem', ascending=False)
    
    # En önemli 15 özellik
    print("En Önemli 15 Özellik (Random Forest):")
    display(importance.head(15))
    
    # Görselleştirme
    fig, ax = plt.subplots(figsize=(10, 8))
    top_15 = importance.head(15)
    sns.barplot(x='onem', y='ozellik', data=top_15, palette='viridis', ax=ax)
    ax.set_xlabel('Önem Skoru')
    ax.set_ylabel('Özellik')
    ax.set_title('Random Forest - En Önemli 15 Özellik')
    plt.tight_layout()
    plt.savefig('../tez_docs/figures/feature_importance.png', dpi=150)
    plt.show()

## 5.5 Precision-Recall Eğrileri

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for idx, (model_adi, model) in enumerate(modeller.items()):
    # Tahmin olasılıkları
    if 'Lojistik' in model_adi:
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = model.predict_proba(X_test)[:, 1]
    
    # PR eğrisi
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    
    ax.plot(recall, precision, color=renkler[idx % len(renkler)], lw=2,
            label=f'{model_adi}')

ax.set_xlabel('Duyarlılık (Recall)')
ax.set_ylabel('Kesinlik (Precision)')
ax.set_title('Precision-Recall Eğrileri')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../tez_docs/figures/precision_recall_curves.png', dpi=150)
plt.show()

## 5.6 Detaylı Sınıflandırma Raporu

In [ ]:
# En iyi model için detaylı rapor
en_iyi_model_adi = sonuc_df['f1'].idxmax()
en_iyi_model = modeller[en_iyi_model_adi]

print(f"=== En İyi Model: {en_iyi_model_adi} ===")
print()

if 'Lojistik' in en_iyi_model_adi:
    y_pred = en_iyi_model.predict(X_test_scaled)
else:
    y_pred = en_iyi_model.predict(X_test)

print(classification_report(y_test, y_pred, 
                            target_names=['Yıldırım Yok', 'Yıldırım Var']))

## 5.7 Sonuç Tablosu (LaTeX için)

In [ ]:
# LaTeX formatında tablo
latex_tablo = sonuc_df.to_latex(
    float_format='%.4f',
    caption='Model Performans Karşılaştırması',
    label='tab:model_sonuclar'
)

print("LaTeX Tablosu:")
print(latex_tablo)

# Kaydet
with open('../tez_docs/tables/model_sonuclar.tex', 'w', encoding='utf-8') as f:
    f.write(latex_tablo)

print("\nTablo kaydedildi: tez_docs/tables/model_sonuclar.tex")

## 5.8 Özet ve Sonuçlar

In [ ]:
print("="*60)
print("YILDIRIM TAHMİNİ PROJESİ - SONUÇ ÖZETİ")
print("="*60)
print()
print(f"Veri Seti Boyutu: {len(df):,} kayıt")
print(f"Özellik Sayısı: {len(ozellikler)}")
print(f"Pozitif Sınıf Oranı: {y.mean()*100:.2f}%")
print()
print("Model Performansları (F1 Skoruna Göre Sıralı):")
print("-"*40)

for model_adi in sonuc_df.sort_values('f1', ascending=False).index:
    f1 = sonuc_df.loc[model_adi, 'f1']
    auc_val = sonuc_df.loc[model_adi, 'roc_auc']
    print(f"  {model_adi}: F1={f1:.4f}, AUC={auc_val:.4f}")

print()
print(f"En İyi Model: {en_iyi_model_adi}")
print("="*60)

In [ ]:
# Tüm figürlerin listesi
import os

print("\nOluşturulan Figürler:")
for f in os.listdir('../tez_docs/figures/'):
    print(f"  - {f}")

---
**Proje Tamamlandı!**

Sonraki adım: `tez_docs/` klasöründeki LaTeX dosyalarını derleyerek PDF oluşturun.